# 實驗：紅線內、綠線外 → 只把地表類型 `gridx01` 改成 5

只改 `source_region_T42.npz` 的 `selected` 那 110 個 T42 格子的 **`gridx01` → 5**（coniferous forest & woodland）。

其他 6 個 nc 檔（`gridx02`、`glfrc01/02`、`slidx`、`grlai01/02`）**完全不動**，原樣複製。

輸出：`../../All_exp_landcover_from_Wei/Landcover5_only/`（完整 7 個 nc 檔）

In [1]:
import os, shutil, hashlib, numpy as np, xarray as xr
ROOT = "/Users/peterchang/Projects/Geography"
SRC = f"{ROOT}/Boundary condition/bc_for_Wei/data/netcdf"
OUT = f"{ROOT}/All_exp_landcover_from_Wei/Landcover5_only"
os.makedirs(OUT, exist_ok=True)
sel = np.load(f"{ROOT}/All_exp_LAI_from_Wei/region/source_region_T42.npz")["selected"]
print("要改的格子：", int(sel.sum()))

ds = xr.open_dataset(f"{SRC}/gridx01.nc", decode_times=False)
new = ds.copy(deep=True)
a = ds["GRIDX01"].values.copy()                    # (1, 64, 128)
a[0, sel] = 5.0
new["GRIDX01"].values[...] = a.astype(ds["GRIDX01"].dtype)
new["GRIDX01"].attrs = dict(ds["GRIDX01"].attrs); new.attrs = dict(ds.attrs)
new.attrs["history"] = f"GRIDX01 set to 5 (coniferous forest & woodland) in {int(sel.sum())} T42 cells (FRONT source region 50%, DES excluded); all other cells unchanged"
new.to_netcdf(f"{OUT}/gridx01.nc"); ds.close()

others = ["gridx02.nc", "glfrc01.y1850.nc", "glfrc02.y1850.nc", "slidx.nc", "grlai01.nc", "grlai02.nc"]
for f in others: shutil.copy(f"{SRC}/{f}", f"{OUT}/{f}")           # 完全不動
print(sorted(os.listdir(OUT)))

要改的格子： 110
['glfrc01.y1850.nc', 'glfrc02.y1850.nc', 'gridx01.nc', 'gridx02.nc', 'grlai01.nc', 'grlai02.nc', 'slidx.nc']


## 檢查

In [2]:
eq = lambda x, y: np.array_equal(x, y, equal_nan=True)
md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
o = xr.open_dataset(f"{SRC}/gridx01.nc", decode_times=False); n = xr.open_dataset(f"{OUT}/gridx01.nc", decode_times=False)
go, gn = o.GRIDX01.values[0], n.GRIDX01.values[0]
print("gridx01：選中格全是 5 :", bool((gn[sel] == 5).all()), "| 其他格與原檔相同 :", eq(gn[~sel], go[~sel]))
print("改前 gridx01 分布（選中格）：", {int(k): int(v) for k, v in zip(*np.unique(go[sel], return_counts=True))}, "→ 5 :", int((gn[sel] == 5).sum()))
print("改變的格子數（值真的不同）:", int((gn != go).sum()), "（選中 110 格中本來就是 5 的有", int((go[sel] == 5).sum()), "格）")
print("座標/時間不變 :", all(eq(o[c].values, n[c].values) for c in ("lon", "lat", "time", "lon_bnds", "lat_bnds")), "| dtype", n.GRIDX01.dtype)
for f in others: print(f"{f:18s} 與原檔 md5 相同 :", md5(f"{SRC}/{f}") == md5(f"{OUT}/{f}"))

gridx01：選中格全是 5 : True | 其他格與原檔相同 : True
改前 gridx01 分布（選中格）： {4: 6, 5: 17, 6: 17, 8: 12, 9: 1, 10: 57} → 5 : 110
改變的格子數（值真的不同）: 93 （選中 110 格中本來就是 5 的有 17 格）
座標/時間不變 : True | dtype float32
gridx02.nc         與原檔 md5 相同 : True
glfrc01.y1850.nc   與原檔 md5 相同 : True
glfrc02.y1850.nc   與原檔 md5 相同 : True
slidx.nc           與原檔 md5 相同 : True
grlai01.nc         與原檔 md5 相同 : True
grlai02.nc         與原檔 md5 相同 : True
